In [ ]:
# Cell 1 — Imports and setup
import os, math, textwrap, random
import numpy as np
import pandas as pd
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, f1_score
from sklearn.model_selection import train_test_split


torch.__version__, torch.cuda.is_available()

In [ ]:
class WaterDataset(Dataset):
    def __init__(self, csv_path):
        super().__init__()
        # Load data to pandas DataFrame
        df = pd.read_csv(csv_path)
        # Convert data to a NumPy array and assign to self.data
        self.data = df.to_numpy()
        
    # Implement __len__ to return the number of data samples
    def __len__(self):
        return self.data.shape[0]
    
    def __getitem__(self, idx):
        features = self.data[idx, :-1]
        # Assign last data column to label
        label = self.data[idx, -1]
        return features, label\
        

# Instructions
# Create an instance of WaterDataset from water_train.csv, assigning it to dataset_train.
# Create dataloader_train based on dataset_train, using a batch size of two and shuffling the samples. The Batch is the size of data samples to be returned in each iteration.
# Get a batch of features and labels from the DataLoader and print them
    
# Create an instance of the WaterDataset
dataset_train = WaterDataset("water_train.csv")

# Create a DataLoader based on dataset_train
dataloader_train = DataLoader(
    dataset=dataset_train,
    batch_size=2,
    shuffle=True
)

# Get a batch of features and labels
features, labels = next(iter(dataloader_train))
print(features, labels)

In [ ]:

# In the .__init__() method, define the three linear layers with dimensions corresponding to the model definition provided and assign them to self.fc1, self.fc2, and self.fc3, respectively.
# In the forward() method, pass the model input x through all the layers, remembering to add activations on top of them, similarly how it's already done for the first layer.

import torch.nn as nn
import torch.nn.functional as F

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        # Define the three linear layers
        self.fc1 = nn.Linear(9, 16)
        self.fc2 = nn.Linear(16, 8)
        self.fc3 = nn.Linear(8, 1)

    def forward(self, x):
        # Pass x through linear layers adding activations
        x = nn.functional.relu(self.fc1(x))
        x = nn.functional.relu(self.fc2(x))
        x = nn.functional.sigmoid(self.fc3(x))
        return x

# Optimizers, training, and evaluation

# Vanishing and exploding gradients

# IMAGE AND CONVOLUTIONAL NEURAL NETWORKS

In [ ]:

# /
# /
# Daily XP
# 604
# Exercise
# Exercise
# Multi-class model evaluation
# Let's evaluate our cloud classifier with precision and recall to see how well it can classify the seven cloud types. In this multi-class classification task it is important how you average the scores over classes. Recall that there are four approaches:

# Not averaging, and analyzing the results per class;
# Micro-averaging, ignoring the classes and computing the metrics globally;
# Macro-averaging, computing metrics per class and averaging them;
# Weighted-averaging, just like macro but with the average weighted by class size.
# Both Precision and Recall are already imported from torchmetrics. It's time to see how well our model is doing!

# Instructions 1/2
# 50 XP
# 1
# Define precision and recall metrics calculated globally on all examples.

# Take Hint (-15 XP)
# 2
# Change your code to compute separate recall and precision metrics for each class and average them with a simple average.
# script.py
# 1234567891011121314
# # Define metrics
# metric_precision = Precision(task=____, num_classes=____, average=____)
# metric_recall = ____

# net.eval()
# with torch.no_grad():
#     for images, labels in dataloader_test:
#         outputs = net(images)
#         _, preds = torch.max(outputs, 1)
#         metric_precision(preds, labels)
#         metric_recall(preds, labels)

# precision = metric_precision.compute()
# recall = metric_recall.compute()
# print(f"Precision: {precision}")
# print(f"Recall: {recall}")
# IPython Shell
# Slides
# In [1]:



# Define metrics
metric_precision = Precision(task="multiclass", num_classes=7, average="micro")
metric_recall = Recall(task="multiclass", num_classes=7, average="micro")

net.eval()
with torch.no_grad():
    for images, labels in dataloader_test:
        outputs = net(images)
        _, preds = torch.max(outputs, 1)
        metric_precision(preds, labels)
        metric_recall(preds, labels)

precision = metric_precision.compute()
recall = metric_recall.compute()
print(f"Precision: {precision}")
print(f"Recall: {recall}")

In [ ]:
# /
# /
# Daily XP
# 1304
# Exercise
# Exercise
# Analyzing metrics per class
# While aggregated metrics are useful indicators of the model's performance, it is often informative to look at the metrics per class. This could reveal classes for which the model underperforms.

# In this exercise, you will run the evaluation loop again to get our cloud classifier's precision, but this time per-class. Then, you will map these score to the class names to interpret them. As usual, Precision has already been imported for you. Good luck!

# Instructions
# 100 XP
# Define a precision metric appropriate for per-class results.
# Calculate the precision per class by finishing the dict comprehension, iterating over the .items() of the .class_to_idx attribute of dataset_test.

# Define precision metric
metric_precision = Precision(
    task="multiclass", num_classes=7, average="none"
)

net.eval()
with torch.no_grad():
    for images, labels in dataloader_test:
        outputs = net(images)
        _, preds = torch.max(outputs, 1)
        metric_precision(preds, labels)
precision = metric_precision.compute()

# Get precision per class
precision_per_class = {
    k: precision[k]
    for k, v in dataset_test.class_to_idx.items()
    in dataset_test.class_to_idx.items()
}
print(precision_per_class)

In [ ]:

# Image classifier training loop
# It's time to train the image classifier! You will use the Net you defined earlier and train it to distinguish between seven cloud types.

# To define the loss and optimizer, you will need to use functions from torch.nn and torch.optim, imported for you as nn and optim, respectively. You don't need to change anything in the training loop itself: it's exactly like the ones you wrote before, with some additional logic to print the loss during training.

# Instructions

# Define the model using your Net class with num_classes set to 7 and assign it to net.
# Define the loss function as cross-entropy loss and assign it to criterion.
# Define the optimizer as Adam, passing it the model's parameters and the learning rate of 0.001, and assign it to optimizer.
# Start the training for-loop by iterating over training images and labels of dataloader_train.

# Take Hint (-30 XP)
# script.py

# Define the model
net = Net(num_classes=7)
# Define the loss function
criterion = nn.CrossEntropyLoss()
# Define the optimizer
optimizer = optim.Adam(net.parameters(), lr=0.001)

for epoch in range(3):
    running_loss = 0.0
    # Iterate over training batches
    for images, labels in dataloader_train:
        optimizer.zero_grad()
        outputs = net(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    epoch_loss = running_loss / len(dataloader_train)
    print(f"Epoch {epoch+1}, Loss: {epoch_loss:.4f}")


# RNN, CNN, Data Augmentation, Transfer Learning, TRANSFORMERS, Optimizers, 

In [ ]:

# Building a forecasting RNN
# It's time to build your first recurrent network! It will be a sequence-to-vector model consisting of an RNN layer with two layers and a hidden_size of 32. After the RNN layer, a simple linear layer will map the outputs to a single value to be predicted.

# The following imports have already been done for you:

import torch
import torch.nn as nn
# Instructions 1/4

# Define the RNN layer passing it the correct values for input_size, hidden_size, num_layers, and batch_first, and assign it to self.rnn.
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        # Define RNN layer
        self.rnn = nn.RNN(
            input_size=1,
            hidden_size=32,
            num_layers=2,
            batch_first=True,
        )
        self.fc = nn.Linear(32, 1)

    def forward(self, x):
        # Initialize first hidden state with zeros
        h0 = torch.zeros(2, x.size(0), 32)
        # Pass x and h0 through recurrent layer
        out, _ = self.rnn(x, h0)  
        # Pass recurrent layer's last output through linear layer
        out = self.fc(out[:, -1, :])
        return out

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        # Define RNN layer
        self.gru = nn.GRU(
            input_size=1,
            hidden_size=32,
            num_layers=2,
            batch_first=True,
        )
        self.fc = nn.Linear(32, 1)

    #Update the RNN model definition in order to obtain a GRU network; assign the GRU layer to self.gru. 

    def forward(self, x):
        h0 = torch.zeros(2, x.size(0), 32)
        out, _ = self.rnn(x, h0)  
        out = self.fc(out[:, -1, :])
        return out

# Cross entropy, training, and evaluation, expanding tensors, squeeze and unsqueeze, training loop, LSTM vs GRU, padding and packing sequences, teacher forcing

what is a batch and a epoch in deep learning?
batch: A batch is a subset of the training dataset that is used to train the model in one iteration. Instead of feeding the entire dataset into the model at once, which can be computationally expensive and memory-intensive, the data is divided into smaller batches. Each batch is processed independently, and the model's weights are updated after each batch.
epoch: An epoch refers to one complete pass through the entire training dataset. During an epoch, the model sees every sample in the dataset once. After each epoch, the model's performance is typically evaluated on a validation set to monitor its learning progress. Multiple epochs are usually required for the model to converge and learn effectively from the data.

In [ ]:
# /
# /
# Daily XP
# 1354
# Exercise
# Exercise
# RNN training loop
# It's time to train the electricity consumption forecasting model!

# You will use the LSTM network you have defined previously, which has been instantiated and assigned to net, as is the dataloader_train you built before. You will also need to use torch.nn which has already been imported as nn.

# In this exercise, you will train the model for only three epochs to make sure the training progresses as expected. Let's get to it!

# Instructions
# 100 XP
# Set up the Mean Squared Error loss and assign it to criterion.
# Reshape seqs to (batch size, sequence length, num features), which in our case is (32, 96, 1), and re-assign the result to seqs.
# Pass seqs to the model to get its outputs.
# Based on previously computed quantities, calculate the loss, assigning it to loss.

net = Net()
# Set up MSE loss
criterion = nn.MSELoss()
optimizer = optim.Adam(
  net.parameters(), lr=0.0001
)

for epoch in range(3):
    for seqs, labels in dataloader_train:
        # Reshape model inputs
        seqs = seqs.view(32, 96, 1)
        # Get model outputs
        outputs = net(seqs)
        # Compute loss
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item()}")


In [ ]:
# /
# /
# Daily XP
# 1454
# Exercise
# Exercise
# Evaluating forecasting models
# It's evaluation time! The same LSTM network that you have trained in the previous exercise has been trained for you for a few more epochs and is available as net.

# Your task is to evaluate it on a test dataset using the Mean Squared Error metric (torchmetrics has already been imported for you). Let's see how well the model is doing!

# Instructions
# 100 XP
# Define the Mean Squared Error metrics and assign it to mse.
# Pass the input sequence to net, and squeeze the result before you assign it to outputs.
# Compute the final value of the test metric assigning it to test_mse.


# u need to use torchmetrics.MeanSquaredError() to define the metric.
# You can call the .squeeze() method on a tensor to get rid of its redundant dimensions.
# Call .compute() on the metric to compute its final value.

# Define MSE metric
mse = torchmetrics.MeanSquaredError()

net.eval()
with torch.no_grad():
    for seqs, labels in dataloader_test:
        seqs = seqs.view(32, 96, 1)
        # Pass seqs to net and squeeze the result
        outputs = net(seqs).squeeze()
        mse(outputs, labels)

# Compute final metric value
test_mse = mse.compute()
print(f"Test MSE: {test_mse}")

In [ ]:
# /
# /
# Daily XP
# 1524
# Exercise
# Exercise
# Generating sequences
# To be able to train neural networks on sequential data, you need to pre-process it first. You'll chunk the data into inputs-target pairs, where the inputs are some number of consecutive data points and the target is the next data point.

# Your task is to define a function to do this called create_sequences(). As inputs, it will receive data stored in a DataFrame, df and seq_length, the length of the inputs. As outputs, it should return two NumPy arrays, one with input sequences and the other one with the corresponding targets.

# As a reminder, here is how the DataFrame df looks like:

#                  timestamp  consumption
# 0      2011-01-01 00:15:00    -0.704319
# ...                    ...          ...
# 140255 2015-01-01 00:00:00    -0.095751
# Instructions
# 100 XP
# Iterate over the range of the number of data points minus the length of an input sequence.
# Define the inputs x as the slice of df from the ith row to the i + seq_lengthth row and the column at index 1.
# Define the target y as the slice of df at row index i + seq_length and the column at index 1.

import numpy as np

def create_sequences(df, seq_length):
    xs, ys = [], []
    # Iterate over data indices
    for i in range(len(df) - seq_length):
      	# Define inputs
        x = df.iloc[i:i + seq_length, 1]
        # Define target
        y = df.iloc[i + seq_length, 1]
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

In [ ]:
# Sequential Dataset
# Good job building the create_sequences() function! It's time to use it to create a training dataset for your model.

# Just like tabular and image data, sequential data is easiest passed to a model through a torch Dataset and DataLoader. To build a sequential Dataset, you will call create_sequences() to get the NumPy arrays with inputs and targets, and inspect their shape. Next, you will pass them to a TensorDataset to create a proper torch Dataset, and inspect its length.

# Your implementation of create_sequences() and a DataFrame with the training data called train_data are available.

# Instructions
# 100 XP
# Call create_sequences(), passing it the training DataFrame and a sequence length of 24*4, assigning the result to X_train, y_train.
# Define dataset_train by calling TensorDataset and passing it two arguments, the inputs and the targets created by create_sequences(), both converted from NumPy arrays to tensors of floats.

import torch
from torch.utils.data import TensorDataset

# Use create_sequences to create inputs and targets
X_train, y_train = create_sequences(train_data, 24*4)
print(X_train.shape, y_train.shape)

# Create TensorDataset
dataset_train = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32)
)
print(len(dataset_train))


# Multiinput and multioutput models with the Functional API

In [ ]:
# /
# /
# Daily XP
# 1774
# Exercise
# Exercise
# Two-input dataset
# Building a multi-input model starts with crafting a custom dataset that can supply all the inputs to the model. In this exercise, you will build the Omniglot dataset that serves triplets consisting of:

# The image of a character to be classified,
# The one-hot encoded alphabet vector of length 30, with zeros everywhere but for a single one denoting the ID of the alphabet the character comes from,
# The target label, an integer between 0 and 963.
# You are provided with samples, a list of 3-tuples comprising an image's file path, its alphabet vector, and the target label. Also, the following imports have already been done for you, so let's get to it!

# from PIL import Image
# from torch.utils.data import DataLoader, Dataset
# from torchvision import transforms
# Instructions 1/4
# 25 XP
# 1
# 2
# 3
# 4
# Assign transform and samples to class attributes with the same names.

class OmniglotDataset(Dataset):
    def __init__(self, transform, samples):
		# Assign transform and samples to class attributes
        self.transform = transform
        self.samples = samples
                    
    def __len__(self):
		# Return number of samples
        return len(self.samples)

    def __getitem__(self, idx):
      	# Unpack the sample at index idx
        img_path, alphabet, label = self.samples[idx]
        img = Image.open(img_path).convert('L')
        # Transform the image 
        img_transformed = self.transform(img)
        return img_transformed, alphabet, label

In [ ]:
# /
# /
# Daily XP
# 1874
# Exercise
# Exercise
# Two-input model
# With the data ready, it's time to build the two-input model architecture! To do so, you will set up a model class with the following methods:

# .__init__(), in which you will define sub-networks by grouping layers; this is where you define the two layers for processing the two inputs, and the classifier that returns a classification score for each class.

# forward(), in which you will pass both inputs through corresponding pre-defined sub-networks, concatenate the outputs, and pass them to the classifier.

# torch.nn is already imported for you as nn. Let's do it!

# Instructions 1/3
# 35 XP
# 1
# 2
# 3
# Define image, alphabet and classifier sub-networks as sequential models, assigning them to self.image_layer, self.alphabet_layer and self.classifier, respectively.

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        # Define sub-networks as sequential models
        self.image_layer = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.MaxPool2d(kernel_size=2),
            nn.ELU(),
            nn.Flatten(),
            nn.Linear(16*32*32, 128)
        )
        self.alphabet_layer = nn.Sequential(
            nn.Linear(30, 8),
            nn.ELU(), 
        )
        self.classifier = nn.Sequential(
            nn.Linear(128 + 8, 964), 
        )
        
    def forward(self, x_image, x_alphabet):
        # Pass the x_image and x_alphabet through appropriate layers
        x_image =  self.image_layer(x_image)
        x_alphabet = self.alphabet_layer(x_alphabet)


In [ ]:
# /
# /
# Daily XP
# 2020
# Exercise
# Exercise
# Two-output Dataset and DataLoader
# In this and the following exercises, you will build a two-output model to predict both the character and the alphabet it comes from based on the character's image. As always, you will start with getting the data ready.

# The OmniglotDataset class you have created before is available for you to use along with updated samples. Let's use it to build the Dataset and the DataLoader.

# The following imports have already been done for you:

# from torch.utils.data import Dataset, DataLoader
# from torchvision import transforms
# Instructions 1/3
# 35 XP
# 1
# 2
# 3
# Print the element of samples at index 100 and examine its structure.


# Print the sample at index 100
# Print the sample at index 100
# Print the sample at index 100
print(samples[100])

# Create dataset_train
dataset_train = OmniglotDataset(
    transform=transforms.Compose([
        transforms.ToTensor(),
      	transforms.Resize((64, 64)),
    ]),
    samples=samples,
)

# Create dataloader_train
dataloader_train = DataLoader(
    dataset=dataset_train,
    batch_size=32,
    shuffle=True,
)

In [ ]:
# /
# /
# Daily XP
# 2086
# Exercise
# Exercise
# Two-output model architecture
# In this exercise, you will construct a multi-output neural network architecture capable of predicting the character and the alphabet.

# Recall the general structure: in the .__init__() method, you define layers to be used in the forward pass later. In the forward() method, you will first pass the input image through a couple of layers to obtain its embedding, which in turn is fed into two separate classifier layers, one for each output.

# torch.nn is already imported under its usual alias, so let's build a model!

# Instructions 1/2
# 50 XP
# 1
# 2
# Define self.classifier_alpha and self.classifier_char as linear layers with input shapes matching the output of image_layer, and output shapes corresponding to the number of alphabets (30) and the number of characters (964), respectively.
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.image_layer = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.MaxPool2d(kernel_size=2),
            nn.ELU(),
            nn.Flatten(),
            nn.Linear(16*32*32, 128)
        )
        # Define the two classifier layers
        self.classifier_alpha = nn.Linear(128, 30)
        self.classifier_char = nn.Linear(128, 964)
        
    def forward(self, x):
        x_image = self.image_layer(x)
        # Pass x_image through the classifiers and return both results
        output_alpha = self.classifier_alpha(x_image)
        output_char = self.classifier_char(x_image)
        return output_alpha, output_char

In [ ]:
# /
# /
# Daily XP
# 2186
# Exercise
# Exercise
# Training multi-output models
# When training models with multiple outputs, it is crucial to ensure that the loss function is defined correctly.

# In this case, the model produces two outputs: predictions for the alphabet and the character. For each of these, there are corresponding ground truth labels, which will allow you to calculate two separate losses: one incurred from incorrect alphabet classifications, and the other from incorrect character classification. Since in both cases you are dealing with a multi-label classification task, the Cross-Entropy loss can be applied each time.

# Gradient descent can optimize only one loss function, however. You will thus define the total loss as the sum of alphabet and character losses.

# Instructions
# 100 XP
# Calculate the alphabet classification loss and assign it to loss_alpha.
# Calculate the character classification loss and assign it to loss_char.
# Compute the total loss as the sum of the two partial losses and assign it to loss.

net = Net()
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.05)

for epoch in range(1):
    for images, labels_alpha, labels_char in dataloader_train:
        optimizer.zero_grad()
        outputs_alpha, outputs_char = net(images)
        # Compute alphabet classification loss
        loss_alpha = criterion(outputs_alpha, labels_alpha)
        # Compute character classification loss
        loss_char = criterion(outputs_char, labels_char)    
        # Compute total loss
        loss = loss_alpha + loss_char
        loss.backward()
        optimizer.step()


In [ ]:
# /
# /
# Daily XP
# 2336
# Exercise
# Exercise
# Multi-output model evaluation
# In this exercise, you will practice model evaluation for multi-output models. Your task is to write a function called evaluate_model() that takes an alphabet-and-character-predicting model as input, runs the evaluation loop, and prints the model's accuracy in the two tasks.

# You can assume that the function will have access to dataloader_test. The following imports have already been run for you:

# import torch
# from torchmetrics import Accuracy
# Once you have implemented evaluate_model(), you will use it in the following exercise!

# Instructions 1/3
# 35 XP
# 1
# 2
# 3
# Define acc_alpha and acc_char as multi-class Accuracy() metrics for the two outputs, alphabets and characters, with the appropriate number of classes each (there are 30 alphabets and 964 characters in the dataset).

def evaluate_model(model):
    # Define accuracy metrics
    acc_alpha = Accuracy(task="multiclass", num_classes=30)
    acc_char = Accuracy(task="multiclass", num_classes=964)

    model.eval()
    with torch.no_grad():
        for images, labels_alpha, labels_char in dataloader_test:
            # Obtain model outputs
            outputs_alpha, outputs_char = model(images)
            _, pred_alpha = torch.max(outputs_alpha, 1)
            _, pred_char = torch.max(outputs_char, 1)
			# Update both accuracy metrics
            acc_alpha.update(pred_alpha, labels_alpha)
            acc_char.update(pred_char, labels_char)
    
    print(f"Alphabet: {acc_alpha.compute()}")
    print(f"Character: {acc_char.compute()}")